In [ ]:
import os, eda_analysis

cfg = eda_analysis.EdaConfig(family="method/contrast")
S   = eda_analysis.notebook_setup(cfg)

# `method/contrast` -- RQ-ii: PTO versus GRPO at matched look-ahead

**The question.** Preference-tree optimization (PTO, a DPO loss over `(chosen, rejected)` pairs)
and group-relative optimization (GRPO, a clipped policy gradient over G sampled completions) are
two ways to spend the same oracle. Held at the same look-ahead depth K, the same MCL, the same
branch width (PTO's `M` = GRPO's `G`) and the same training rubric, which one produces the better
therapist?

**The comparison is at a matched ITERATION, not a matched price.** Two arms at the same iteration
index have not spent the same GPU-hours -- PTO's preference build is a phase GRPO does not have,
and a K=5 step costs more than a K=0 one. That axis is `compute/cost`'s, and the two readings can
disagree; this family is the per-iteration one and says so on every artifact.

**THE SIGN CONVENTION, used everywhere below.**

* `mean_delta = score(PTO) - score(GRPO)` -- positive means **PTO scored HIGHER**. Higher, not
  better.
* `gain = sign_of(metric) * mean_delta` -- positive means **PTO was BETTER**, on every
  instrument, including MICI, which counts MI-INCONSISTENT behaviour and is lower-is-better.
* CI ends are swapped when the sign flips, so `gain_ci_lo <= gain_ci_hi` on every row.

**The pairing unit is `persona_id`.** The same 96 personas face both methods, so each contrast is
repeated-measures: subtract within persona, analyse the 96 deltas. Persona variance dominates the
method difference, so an unpaired test on this data answers a different and much noisier question.
`stats.paired_arrays` joins on `persona_id`; nothing here pairs by row order.

**Multiplicity.** Holm-Bonferroni **within each grader**, over that grader's rows in the endpoint
table (every instrument x every K). Concatenated per-grader tables are not jointly corrected.

**Both endpoints matter.** The matched-state row is one point on two trajectories. Section 4 walks
every shared state, and `arms/outcomes` reports each arm's own best state -- if an arm regressed
after its peak, the matched-endpoint row is not the whole story.

In [ ]:
# ---------------------------------------------------------------------------
# Imports, both graders, the method pairs, and the guards that let this notebook
# render with NO DATA on disk.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eda_analysis import constants, data, exports, plotting, stats

SEED = S.CFG.boot_seed
FOCUS = S.CFG.focus_metric

ALL = data.scores_by_judge(S.ARMS, rep=S.CFG.judge_rep, attach_persona=S.CFG.attach_persona)
if ALL.empty and not S.SCORES.empty:
    ALL = S.SCORES
HAVE = not ALL.empty

JUDGES = sorted(ALL["judge"].unique()) if HAVE else [S.JUDGE]
METRICS = [m for m in constants.METRIC_ORDER if HAVE and m in set(ALL["metric"].unique())]
ARMS = sorted(ALL["arm_label"].unique()) if HAVE else []
PALETTE = plotting.arm_palette(ARMS) if ARMS else {}

NO_DATA = ("NO DATA YET -- no scored conversations for these arms. Generate a PTO arm and a GRPO "
           "arm at the same K, run notebooks/scoring/Run_Eval.ipynb, and re-render.")
NO_PAIR = ("NO METHOD CONTRAST AVAILABLE -- this needs a PTO arm and a GRPO arm at the SAME "
           "look-ahead depth. Only one method is on disk at any given K.")

exports.reset_results()
exports.save_provenance(S.CFG, ALL)


def placeholder(name, message=NO_DATA, group=None, caption=None):
    fig = plt.figure(figsize=(7.6, 1.9))
    ax = fig.add_subplot(111)
    ax.axis("off")
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=8.5,
            color="#777777", wrap=True)
    exports.save_fig(fig, name, group=group, caption=caption or message)
    plt.close(fig)


def method_pairs(arms):
    # One contrast per look-ahead depth that has BOTH methods. K is the matched variable:
    # comparing PTO at one K with GRPO at another would confound the two levers this
    # experiment exists to separate.
    by_k = {}
    for a in arms:
        by_k.setdefault(a.k, {}).setdefault(a.method, set()).add(a.label)
    out = []
    for k in sorted(by_k):
        methods = by_k[k]
        if "PTO" not in methods or "GRPO" not in methods:
            continue
        for arm_pto in sorted(methods["PTO"]):
            for arm_grpo in sorted(methods["GRPO"]):
                out.append({"k": k, "arm_a": arm_pto, "arm_b": arm_grpo,
                            "label": f"PTO - GRPO @ K={k}"})
    return out


PAIRS = method_pairs(S.ARMS)
labels = [a.label for a in S.ARMS]
if len(set(labels)) != len(labels):
    print("WARNING: two arms share a display label (they differ only in something the label "
          "elides -- MCL, branch width, rubric). Key on experiment_name if that matters here.")

print(f"judges : {JUDGES}")
print(f"metrics: {METRICS or '(none scored)'}")
print(f"pairs  : {[p['label'] for p in PAIRS] or '(none -- need both methods at one K)'}")
if not HAVE:
    print(NO_DATA)
elif not PAIRS:
    print(NO_PAIR)

## 1. The contrast machinery

Identical in shape to `lookahead/reward`'s -- deliberately recomputed here rather than imported
from a sibling's rendered tables, because families in this EDA are self-contained. `cell_scores`
narrows to exactly one (grader, arm, model state, instrument) cell, which is what
`stats.paired_arrays` requires: it raises on a frame that still holds several instruments, since
pairing that would build a cross product instead of pairs.

`contrast_row` is the only place a difference is computed here, so the sign convention
(**`PTO - GRPO`**) lives in one place.

In [ ]:
CONTRAST_COLS = ["judge", "contrast", "k", "metric", "instrument", "iteration",
                 "arm_a", "arm_b", "n", "mean_delta", "ci_lo", "ci_hi", "dz",
                 "gain", "gain_ci_lo", "gain_ci_hi", "gain_dz", "improved", "effect",
                 "t", "p", "p_holm", "stars", "sign"]


def cell_scores(df, judge, arm, state, metric):
    # ONE (grader, arm, model state, instrument) cell, as persona_id + score.
    m = df[(df["judge"] == judge) & (df["arm_label"] == arm)
           & (df["iteration"] == int(state)) & (df["metric"] == metric)]
    return m[["persona_id", "score"]].dropna(subset=["score"])


def states_for(df, judge, arm, metric):
    sub = df[(df["judge"] == judge) & (df["arm_label"] == arm) & (df["metric"] == metric)]
    return set(int(s) for s in sub["iteration"].unique())


def common_states(df, judge, metric, arms):
    common = None
    for arm in arms:
        s = states_for(df, judge, arm, metric)
        common = s if common is None else (common & s)
    return sorted(common or [])


def matched_state(df, judge, arms, metrics):
    # Highest TRAINED state both arms have scored on EVERY instrument, so all the metrics of
    # one row are read at the same iteration index. State 0 is the shared untrained base and
    # is reported separately as a placebo, never as an endpoint.
    common = None
    for arm in arms:
        for metric in metrics:
            s = states_for(df, judge, arm, metric)
            common = s if common is None else (common & s)
    trained = sorted(x for x in (common or []) if x > 0)
    return trained[-1] if trained else None


def contrast_row(df, judge, metric, state, arm_a, arm_b, **extra):
    # SIGN: arm_a - arm_b, with arm_a always the PTO arm. Positive mean_delta = PTO scored
    # HIGHER; `gain` re-signs it so positive is BETTER on every instrument. Paired on persona_id.
    a = cell_scores(df, judge, arm_a, state, metric)
    b = cell_scores(df, judge, arm_b, state, metric)
    if a.empty or b.empty:
        return None
    va, vb = stats.paired_arrays(a, b, on="persona_id", value="score")
    if va.size == 0:
        return None
    row = stats.orient_contrast(stats.paired_contrast(va, vb, seed=SEED), metric)
    row.update({"judge": judge, "metric": metric, "instrument": constants.short_label(metric),
                "iteration": int(state), "arm_a": arm_a, "arm_b": arm_b,
                "effect": stats.effect_label(row.get("gain_dz"))})
    row.update(extra)
    return row


def delta_rows(df, judge, metric, state, arm_a, arm_b):
    # Per-persona PTO - GRPO delta, one row per persona. The by-state figure bootstraps THESE,
    # so its band is a paired interval.
    a = cell_scores(df, judge, arm_a, state, metric)
    b = cell_scores(df, judge, arm_b, state, metric)
    if a.empty or b.empty:
        return pd.DataFrame({"persona_id": pd.Series(dtype="int64"),
                             "score": pd.Series(dtype="float64")})
    m = a.merge(b, on="persona_id", how="inner", suffixes=("_a", "_b"))
    return pd.DataFrame({"persona_id": m["persona_id"].to_numpy(),
                         "score": (m["score_a"] - m["score_b"]).to_numpy(dtype=float)})


def tidy_by_judge(rows):
    # Holm-Bonferroni WITHIN each grader: the family is that grader's rows and nothing else.
    if not rows:
        return pd.DataFrame()
    raw = pd.DataFrame([r for r in rows if r is not None])
    if raw.empty:
        return pd.DataFrame()
    parts = [stats.summarize_contrasts(g) for _, g in raw.groupby("judge", sort=True)]
    out = pd.concat(parts, ignore_index=True)
    lead = [c for c in CONTRAST_COLS if c in out.columns]
    return out[lead + [c for c in out.columns if c not in lead]]


print("contrast helpers ready")

## 2. PTO versus GRPO at the matched endpoint

One row per (grader, K, instrument) at the highest trained state both arms have scored. `n` is
the number of personas that survived the inner join -- the honest sample size, not 96 by
assumption.

The state-0 placebo is the same comparison at the untrained base, where both arms are the *same
policy* and only the sampled conversations differ. It bounds what "no method difference" looks
like on each instrument under each grader.

In [ ]:
rows = []
placebo = []
matched = {}
if HAVE and PAIRS and METRICS:
    for judge in JUDGES:
        for pair in PAIRS:
            arms = [pair["arm_a"], pair["arm_b"]]
            state = matched_state(ALL, judge, arms, METRICS)
            matched[(judge, pair["label"])] = state
            if state is None:
                continue
            for metric in METRICS:
                rows.append(contrast_row(ALL, judge, metric, state, pair["arm_a"], pair["arm_b"],
                                         contrast=pair["label"], k=pair["k"]))
                if 0 in common_states(ALL, judge, metric, arms):
                    placebo.append(contrast_row(
                        ALL, judge, metric, 0, pair["arm_a"], pair["arm_b"],
                        contrast=pair["label"] + " @ base", k=pair["k"]))

M_ENDPOINT = tidy_by_judge(rows)
M_PLACEBO = tidy_by_judge(placebo)

exports.save_table(
    M_ENDPOINT, "method_contrast_endpoint",
    caption=("RQ-ii at the matched endpoint. One row per grader x look-ahead depth x instrument, "
             "paired on persona_id at the highest trained model state both arms have scored. "
             "SIGN: mean_delta = PTO - GRPO (higher, not better); gain = sign_of(metric) * "
             "mean_delta, so positive means PTO was BETTER on every instrument including MICI. "
             "CIs are 2,000-resample percentile bootstraps of the paired mean (seed=BOOT_SEED); "
             "p_holm is corrected WITHIN each grader. Matched by ITERATION, not by spend -- see "
             "compute/cost for the budget-matched reading."))
exports.save_table(
    M_PLACEBO, "method_contrast_placebo_base",
    caption=("PLACEBO: the same PTO - GRPO comparison at model state 0, where both arms ARE the "
             "same untrained policy and only the sampled conversations differ. Non-zero gain "
             "here is sampling noise and bounds what 'no method difference' looks like. Same "
             "sign convention and pairing unit as the endpoint table."))

if M_ENDPOINT.empty:
    print(NO_DATA if not (HAVE and PAIRS) else "no matched state shared by a method pair yet")
else:
    print({k: v for k, v in matched.items()})
    display(M_ENDPOINT[M_ENDPOINT["metric"] == FOCUS])

## 3. Forest plot of the endpoint contrasts

One dot and CI per (grader, K, instrument). The x axis is the **raw** paired difference
`PTO - GRPO`, so the plotted number matches the table; only the colour is oriented, which is why a
green dot on MICI sits at a negative x. Grey means the interval covers zero.

In [ ]:
if M_ENDPOINT.empty:
    placeholder("method_contrast_forest", caption="No method contrast to plot yet. " + NO_DATA)
else:
    forest = M_ENDPOINT.copy()
    forest["label"] = (forest["contrast"] + "  |  " + forest["instrument"]
                       + "  [" + forest["judge"] + "]")
    forest = forest.sort_values(["judge", "k", "metric"])
    fig = plotting.contrast_forest(
        forest, label_col="label", value_col="mean_delta", lo_col="ci_lo", hi_col="ci_hi",
        annot_col="dz", metric_col="metric",
        title="RQ-ii: PTO - GRPO at the matched endpoint",
        xlabel="paired difference PTO - GRPO (95% bootstrap CI)")
    if fig is not None:
        exports.save_fig(
            fig, "method_contrast_forest",
            caption=("Paired PTO - GRPO contrasts at the matched endpoint, one row per grader x "
                     "K x instrument, pairing unit persona_id. The x axis is the RAW difference; "
                     "colour is oriented by instrument, so on MICI (lower better) a green dot "
                     "lies at a negative x. Grey = the 95% bootstrap CI includes zero. Matched by "
                     "iteration, not by spend."))
        plt.close(fig)
    print(f"forest rendered with {len(forest)} rows")

## 4. The method gap across model states

The paired `PTO - GRPO` delta on the training-reward axis at **every** state both arms have
scored. One endpoint is one point on two trajectories that need not be parallel: an arm that
peaks early and regresses can lead at one state and trail at the next, and this is where that is
visible.

The band bootstraps the per-persona deltas, so unlike the unpaired bands in `arms/outcomes` it
**is** a paired interval.

In [ ]:
records = []
if HAVE and PAIRS:
    for judge in JUDGES:
        for pair in PAIRS:
            for state in common_states(ALL, judge, FOCUS, [pair["arm_a"], pair["arm_b"]]):
                d = delta_rows(ALL, judge, FOCUS, state, pair["arm_a"], pair["arm_b"])
                if d.empty:
                    continue
                records.append(d.assign(judge=judge, iteration=int(state), metric=FOCUS,
                                        contrast=pair["label"]))

DELTA_LONG = (pd.concat(records, ignore_index=True) if records
              else pd.DataFrame(columns=["persona_id", "score", "judge", "iteration",
                                         "metric", "contrast"]))

by_state_rows = []
if not DELTA_LONG.empty:
    sign = int(constants.sign_of(FOCUS))
    for (judge, contrast, state), g in DELTA_LONG.groupby(["judge", "contrast", "iteration"]):
        x = g["score"].to_numpy(dtype=float)
        lo, hi = stats.bootstrap_ci(x, np.mean, seed=SEED)
        by_state_rows.append({"judge": judge, "contrast": contrast, "iteration": int(state),
                              "metric": FOCUS, "n": int(x.size),
                              "mean_delta": float(np.mean(x)), "ci_lo": lo, "ci_hi": hi,
                              "gain": sign * float(np.mean(x)),
                              "gain_ci_lo": lo if sign > 0 else -hi,
                              "gain_ci_hi": hi if sign > 0 else -lo,
                              "dz": stats.cohens_dz(x, np.zeros_like(x)), "sign": sign})

M_BY_STATE = (pd.DataFrame(by_state_rows).sort_values(["judge", "contrast", "iteration"])
              .reset_index(drop=True) if by_state_rows else pd.DataFrame())
exports.save_table(
    M_BY_STATE, "method_contrast_by_state",
    caption=(f"The paired PTO - GRPO contrast on {FOCUS} at EVERY model state both arms have "
             f"scored, per grader. mean_delta = PTO - GRPO; gain is signed so positive means PTO "
             f"was better. CIs bootstrap the per-persona DELTAS (seed=BOOT_SEED) and are "
             f"therefore paired intervals. State 0 is the shared untrained base. No multiplicity "
             f"correction across states -- this is one trajectory, not a family of tests."))

for judge in JUDGES:
    name = f"method_gap_by_state_{judge}"
    sub = DELTA_LONG[DELTA_LONG["judge"] == judge] if not DELTA_LONG.empty else DELTA_LONG
    if sub.empty:
        placeholder(name, message=(NO_DATA if not PAIRS else NO_PAIR),
                    caption=f"No method contrast for grader {judge}.")
        continue
    fig = plotting.score_trajectory(
        sub, metric=FOCUS, metric_col="metric", x="iteration", y="score", arm_col="contrast",
        base_value=None, title=f"PTO - GRPO on {constants.short_label(FOCUS)} ({judge})",
        xlabel="model state", ylabel=f"paired delta in {FOCUS}  (PTO - GRPO)")
    plotting.add_base_line(fig.axes[0], 0.0, label="no difference")
    legend = fig.axes[0].get_legend()
    if legend is not None:
        legend.set_title("contrast")
    exports.save_fig(
        fig, name,
        caption=(f"Mean PAIRED delta ({FOCUS}, PTO - GRPO) by model state, grader {judge}, "
                 f"pairing unit persona_id. The band bootstraps the per-persona deltas "
                 f"(seed=BOOT_SEED) and is a paired interval. Dotted line = no difference; state "
                 f"0 is the shared untrained base. Positive = PTO scored higher ({FOCUS} is "
                 f"higher-is-better). The x axis is iterations, which is NOT a fixed unit of "
                 f"spend -- see compute/cost."))
    plt.close(fig)
print(f"{len(M_BY_STATE)} by-state rows")

## 5. The distributions behind the matched endpoint

Both methods' per-persona scores at the matched state, on the training-reward axis, with the
untrained base level marked. The same 96 personas are in every box, so heavy overlap is entirely
compatible with the decisive paired difference the forest may show -- the boxes carry persona
variance that the contrast has already differenced out.

In [ ]:
for judge in JUDGES:
    for pair in PAIRS or []:
        state = matched.get((judge, pair["label"]))
        name = f"endpoint_distribution_K{pair['k']}_{judge}"
        if state is None:
            continue
        frame = ALL[(ALL["judge"] == judge) & (ALL["metric"] == FOCUS)
                    & (ALL["iteration"] == int(state))
                    & (ALL["arm_label"].isin([pair["arm_a"], pair["arm_b"]]))]
        if frame.empty:
            placeholder(name, caption=f"{NO_DATA} (grader {judge}, K={pair['k']})")
            continue
        base_level = ALL[(ALL["judge"] == judge) & (ALL["metric"] == FOCUS)
                         & (ALL["iteration"] == 0)]["score"].mean()
        # arm_distribution overlays a seaborn stripplot, whose JITTER draws from numpy's global
        # RNG and accepts no seed argument -- so without this, two renders of identical data
        # produce different PNGs and every tracked figure churns in git. Its bootstrap CI is
        # already seeded inside plotting; the jitter is the one draw that callsite cannot reach.
        np.random.seed(SEED)
        fig = plotting.arm_distribution(
            frame, metric=FOCUS, metric_col="metric", arm_col="arm_label", palette=PALETTE,
            title=f"{constants.short_label(FOCUS)} at state {state}, K={pair['k']} ({judge})",
            ylabel="grader score")
        plotting.add_base_line(fig.axes[0], base_level, label="base")
        exports.save_fig(
            fig, name,
            caption=(f"Per-persona {FOCUS} for both methods at the matched state {state}, K="
                     f"{pair['k']}, grader {judge}. Box = median, diamond = mean with an "
                     f"UNPAIRED 95% bootstrap CI (seed=BOOT_SEED), dotted line = the untrained "
                     f"base level. The same 96 personas are in both boxes, so their overlap does "
                     f"not bound the paired difference reported in the forest."))
        plt.close(fig)

if not PAIRS:
    placeholder("endpoint_distribution", message=(NO_DATA if not HAVE else NO_PAIR),
                caption="No matched method pair to draw distributions for.")
print("endpoint distributions rendered")

## 6. Number ledger and index

The citable form of RQ-ii, per grader and per K: the sign convention, the matched state, and the
gain with its CI on the training-reward axis. Nothing is pooled across graders.

In [ ]:
values = {
    "rqii.sign_convention": {
        "value": "mean_delta = score(PTO) - score(GRPO); gain = sign_of(metric) * mean_delta",
        "source": "tables/method_contrast_endpoint.md",
        "note": "positive gain = PTO was BETTER, on every instrument"},
    "rqii.pairing_unit": {"value": "persona_id", "source": "",
                          "note": "repeated measures; the same 96 personas face both methods"},
    "rqii.multiplicity": {"value": "Holm-Bonferroni within grader",
                          "source": "tables/method_contrast_endpoint.md",
                          "note": "family = that grader's rows (instruments x look-ahead depths)"},
    "rqii.matched_on": {"value": "iteration", "source": "",
                        "note": ("NOT matched on spend: PTO's preference build is a phase GRPO "
                                 "does not have, so the same iteration is a different price. "
                                 "compute/cost holds the budget-matched reading.")},
    "rqii.pairs": {"value": [p["label"] for p in PAIRS], "source": "", "note": ""},
    "rqii.matched_states": {"value": {f"{j} | {c}": v for (j, c), v in matched.items()},
                            "source": "tables/method_contrast_endpoint.md",
                            "note": "highest trained state both arms scored on every instrument"},
}
if not M_ENDPOINT.empty:
    for _, r in M_ENDPOINT[M_ENDPOINT["metric"] == FOCUS].iterrows():
        key = f"rqii.{r['judge']}.K{int(r['k'])}.{FOCUS}"
        values[key] = {
            "value": float(r["gain"]),
            "source": "tables/method_contrast_endpoint.md",
            "note": (f"gain (positive = PTO better) at state {int(r['iteration'])}, "
                     f"n={int(r['n'])} personas, 95% CI "
                     f"[{float(r['gain_ci_lo']):.3f}, {float(r['gain_ci_hi']):.3f}], "
                     f"dz={float(r['gain_dz']):.3f}, p_holm={float(r['p_holm']):.4f}")}

exports.save_numbers(
    "method_contrast", values,
    caption=("Citable RQ-ii numbers, per grader and look-ahead depth, with the sign convention, "
             "the pairing unit and the matched-on axis stated alongside."))
print(exports.build_index())